# Model Analysis: Embedding Quality & Optimization
Analyze MERT embeddings, PCA variance, and quantization effects for AURA.

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
from models.audio_encoder import MERTEncoder
from models.embedding_optimizer import EmbeddingOptimizer
from sklearn.decomposition import PCA
%matplotlib inline

In [ ]:
# Load a subset of embeddings (or generate new ones)
# For demo, we'll load from cache if available
import pickle
try:
    with open('../experiments/results/iteration1_baseline/embeddings.pkl', 'rb') as f:
        data = pickle.load(f)
    embeddings = data['embeddings']
    print(f"Loaded {embeddings.shape[0]} embeddings of dim {embeddings.shape[1]}")
except FileNotFoundError:
    print("Generate embeddings first by running experiment_runner.py")
    embeddings = None

In [ ]:
if embeddings is not None:
    # PCA variance analysis
    pca = PCA()
    pca.fit(embeddings[:10000])
    cumsum = np.cumsum(pca.explained_variance_ratio_)
    plt.figure(figsize=(10,6))
    plt.plot(cumsum)
    plt.xlabel('Number of components')
    plt.ylabel('Cumulative explained variance')
    plt.grid()
    plt.title('PCA Explained Variance')
    # Mark 95% threshold
    n_95 = np.argmax(cumsum >= 0.95) + 1
    plt.axhline(y=0.95, color='r', linestyle='--')
    plt.axvline(x=n_95, color='g', linestyle='--')
    plt.text(n_95+5, 0.8, f'95% at {n_95} comps', fontsize=12)
    plt.show()

In [ ]:
# Quantization effect
if embeddings is not None:
    optimizer = EmbeddingOptimizer(768, 128)
    optimizer.fit_pca(embeddings[:5000])
    reduced = optimizer.transform(embeddings[:100])
    quantized = optimizer.quantize(reduced)
    dequantized = optimizer.dequantize(quantized)
    
    # Compare error
    mse = np.mean((reduced - dequantized)**2)
    print(f"MSE after quantization/dequantization: {mse:.6f}")
    print(f"Memory reduction: {reduced.nbytes / quantized.nbytes:.2f}x")